In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning) 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
sns.set_style("whitegrid")

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# ------------------------------------------------
# Генератор датасета Coffe2Go
# ------------------------------------------------

np.random.seed(42)
n = 5000

# Признаки
hour = np.random.randint(0, 24, n)
dayofweek = np.random.randint(0, 7, n)
cust_dist = np.round(np.random.uniform(0.5, 15.0, n), 2)
items_count = np.random.randint(1, 9, n)
total_price = np.round(np.random.uniform(100, 1500, n), 2)
barista_exp = np.random.randint(1, 61, n)
promo = np.random.binomial(1, 0.3, n)
rain = np.random.binomial(1, 0.4, n)

# ------------------- Таргет 1: time_to_serve (регрессия) -------------------
# Базовое время: от числа позиций, стажа (обратная связь), 
# эффект дождя и часа пик (8-10, 17-19)
base_time = 30 + items_count * 30 - barista_exp * 1.5 + rain * 40
peak_morning = ((hour >= 8) & (hour <= 10)).astype(int) * 30
peak_evening = ((hour >= 17) & (hour <= 19)).astype(int) * 20
noise_time = np.random.normal(0, 15, n)
time_to_serve = np.clip(base_time + peak_morning + peak_evening + noise_time, 30, 600).astype(int)

# ------------------- Таргет 2: good_review (сбалансированная классификация) -------------------
# Вероятность хорошего отзыва падает при долгом ожидании и растёт от стажа и наличия акции
logit_review = (
    -0.01 * time_to_serve
    + 0.03 * barista_exp
    + 0.5 * promo
    - 0.4 * rain
    + np.random.normal(0, 0.5, n)
)
prob_review = 1 / (1 + np.exp(-logit_review))
# Подгоняем порог, чтобы было примерно 50/50 (сбалансированный)
threshold_review = np.median(prob_review)  # 0.5 по медиане
good_review = (prob_review > threshold_review).astype(int)

# ------------------- Таргет 3: repeat_visit (несбалансированная классификация) -------------------
# Повторный визит редок (20-30%). Зависит от качества обслуживания, цены, расстояния
logit_repeat = (
    -0.005 * time_to_serve
    + 0.02 * barista_exp
    - 0.3 * cust_dist
    - 0.001 * total_price
    + 0.3 * promo
    + np.random.normal(0, 0.6, n)
)
prob_repeat = 1 / (1 + np.exp(-logit_repeat))
# Подгоняем порог, чтобы repeat_visit ≈ 0.25 (25% положительных)
threshold_repeat = np.percentile(prob_repeat, 75)
repeat_visit = (prob_repeat > threshold_repeat).astype(int)

# ------------------- Собираем DataFrame -------------------
df = pd.DataFrame({
    'hour': hour,
    'dayofweek': dayofweek,
    'cust_dist': cust_dist,
    'items_count': items_count,
    'total_price': total_price,
    'barista_exp': barista_exp,
    'promo': promo,
    'rain': rain,
    'time_to_serve': time_to_serve,
    'good_review': good_review,
    'repeat_visit': repeat_visit
})

# Проверка баланса
print("Баланс good_review:", df['good_review'].value_counts(normalize=True).round(2))
print("Баланс repeat_visit:", df['repeat_visit'].value_counts(normalize=True).round(2))
print("\nСтатистика time_to_serve:\n", df['time_to_serve'].describe())
print("\nРазмер датасета:", df.shape)


Баланс good_review: 1    0.5
0    0.5
Name: good_review, dtype: float64
Баланс repeat_visit: 0    0.75
1    0.25
Name: repeat_visit, dtype: float64

Статистика time_to_serve:
 count    5000.000000
mean      143.532400
std        74.724315
min        30.000000
25%        80.000000
50%       141.000000
75%       203.000000
max       347.000000
Name: time_to_serve, dtype: float64

Размер датасета: (5000, 11)


In [5]:
df

,hour,dayofweek,cust_dist,items_count,total_price,barista_exp,promo,rain,time_to_serve,good_review,repeat_visit
0,6,1,13.39,4,101.58,51,1,1,127,1,0
1,19,1,10.54,4,1351.93,58,0,1,131,1,0
2,14,1,4.03,2,535.46,34,1,0,47,1,1
3,10,3,14.01,2,536.09,59,0,0,40,1,0
4,7,2,2.02,7,419.32,21,1,1,244,0,1
...,...,...,...,...,...,...,...,...,...,...,...
4995,11,1,6.84,1,1177.36,51,0,0,30,1,0
4996,10,5,4.21,7,471.26,5,0,0,272,0,0
4997,14,1,2.24,8,1348.13,22,0,0,227,0,1
4998,15,3,13.72,6,200.00,9,0,0,219,0,0


In [6]:
# df.to_csv('data/coffe2go.csv', index = False)